# Drift-Aware Resource Prediction Pipeline
## Complete Pipeline: Raw Data → Sequences (Steps 1–4)

### Pipeline
```
Step 1: Load raw long-format CSVs from Google Drive
Step 2: Pivot long → wide format + assign container IDs
Step 3: Split (60/20/20) + Normalize + Feature Engineering
Step 4: Sliding window sequence generation (.npy output)
```

### Key Design Decisions
- **No data leakage:** normalization stats calculated on training split only
- **Per-container grouping:** lag/rolling features computed within each container
- **RAM-optimized:** float32, in-place ops, immediate disk writes, gc after each step
- **No new folders created:** uses existing `data/sequences` directory

---
**Author:** Team-Dracasys | **Date:** 2026-05-16 | **Version:** RAM-OPTIMIZED

# STEP 0: Memory Monitoring Setup

In [ ]:
import psutil
import os
import gc

def get_memory_usage():
    """Get current memory usage in MB."""
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / 1024 / 1024

def log_memory(label: str):
    """Log memory usage with label."""
    gc.collect()  # Force garbage collection
    mem = get_memory_usage()
    print(f"💾 [{label}] RAM: {mem:.1f} MB")
    return mem

print("✓ Memory monitoring setup complete")
initial_mem = log_memory("Initial")

# STEP 1: Mount Google Drive & Verify Access

In [ ]:
import sys
try:
    from google.colab import drive
    IN_COLAB = True
    print("✓ Running in Google Colab")
except ImportError:
    IN_COLAB = False
    print("⚠ Running locally (not Google Colab)")

print(f"Python version: {sys.version}")

In [ ]:
if IN_COLAB:
    from google.colab import drive
    from pathlib import Path
    
    print("Mounting Google Drive...")
    drive.mount('/content/drive')
    print("\n✓ Google Drive mounted!")
    
    raw_path = Path('/content/drive/My Drive/raw')
    
    if raw_path.exists():
        print(f"✓ Found raw folder at: {raw_path}")
        all_csv_files = list(raw_path.glob('**/*.csv'))
        print(f"✓ Found {len(all_csv_files)} CSV files")
    else:
        print(f"❌ raw folder not found at: {raw_path}")
else:
    print("To use this notebook, please run it in Google Colab")
    print("Open in Colab: https://colab.research.google.com")

# STEP 2: Setup Imports & Configure Paths

In [ ]:
import pandas as pd
import numpy as np
import logging
from pathlib import Path
from typing import List, Dict, Tuple
import json
import warnings
warnings.filterwarnings('ignore')

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print("✓ All imports successful")

In [ ]:
# Setup Paths
gd_raw_path    = Path('/content/drive/My Drive/raw')
output_base    = Path('/content/processed_data')
processed_path = output_base / 'processed'
merged_path    = output_base / 'merged'
sequences_path = output_base / 'sequences'

print("📁 Path Configuration:")
print(f"  Google Drive Input:  {gd_raw_path}")
print(f"  Processed Output:    {processed_path}")
print(f"  Merged Output:       {merged_path}")
print(f"  Sequences Output:    {sequences_path}")

log_memory("After setup")

# STEP 1 + 2: Load Raw Data & Pivot to Wide Format

In [ ]:
def load_data_from_drive_optimized(raw_path: Path) -> pd.DataFrame:
    """Load CSV files and pivot from long format to wide format.

    Raw CSV format (long):
        timestamp | cmdb_id | kpi_name | value

    Output format (wide):
        timestamp | cmdb_id | new_container_id | metric_1 | metric_2 | ...
    """
    logger.info(f"Loading data from Google Drive (RAM-optimized)...")

    csv_files = sorted(raw_path.glob('**/*.csv'))
    logger.info(f"Found {len(csv_files)} CSV files")

    if not csv_files:
        logger.error("No CSV files found!")
        return None

    # Target metrics to keep
    target_metrics = [
        'container_cpu_usage_seconds_total',
        'container_cpu_system_seconds_total',
        'container_cpu_user_seconds_total',
        'container_memory_usage_bytes',
        'container_memory_working_set_bytes',
        'container_memory_rss',
        'container_memory_cache'
    ]

    all_dfs = []
    for idx, csv_file in enumerate(csv_files, 1):
        try:
            df = pd.read_csv(csv_file)
            # Keep only rows for target metrics
            if 'kpi_name' in df.columns:
                df = df[df['kpi_name'].isin(target_metrics)]
            all_dfs.append(df)
            if idx % 10 == 0:
                logger.info(f"  Loaded {idx}/{len(csv_files)} files")
        except Exception as e:
            logger.warning(f"  Error loading {csv_file.name}: {e}")

    if not all_dfs:
        logger.error("Failed to load any CSV files")
        return None

    # Concatenate all long-format data
    combined = pd.concat(all_dfs, ignore_index=True)
    del all_dfs
    gc.collect()
    logger.info(f"Combined long-format rows: {len(combined):,}")

    # PIVOT: long format → wide format (one column per metric)
    logger.info("Pivoting long → wide format...")
    pivoted = combined.pivot_table(
        index=['timestamp', 'cmdb_id'],
        columns='kpi_name',
        values='value',
        aggfunc='first'
    ).reset_index()
    pivoted.columns.name = None
    del combined
    gc.collect()

    # Assign new_container_id (alphabetically sorted cmdb_id → container_1..N)
    unique_cmdb = sorted(pivoted['cmdb_id'].unique())
    cmdb_to_id = {cmdb: f"container_{i+1}" for i, cmdb in enumerate(unique_cmdb)}
    pivoted['new_container_id'] = pivoted['cmdb_id'].map(cmdb_to_id)
    logger.info(f"Assigned IDs to {len(unique_cmdb)} unique containers")

    # Convert metrics to float32 to save RAM
    metric_cols = [c for c in pivoted.columns if c not in ['timestamp', 'cmdb_id', 'new_container_id']]
    for col in metric_cols:
        pivoted[col] = pd.to_numeric(pivoted[col], errors='coerce').astype(np.float32)

    # Sort by timestamp
    pivoted.sort_values(['timestamp', 'cmdb_id'], inplace=True)
    pivoted.reset_index(drop=True, inplace=True)

    logger.info(f"✓ Final shape: {pivoted.shape[0]:,} rows × {pivoted.shape[1]} columns")
    logger.info(f"  Columns: {list(pivoted.columns)}")
    return pivoted


df = load_data_from_drive_optimized(gd_raw_path)

if df is not None:
    print(f"✓ Data loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"  Columns: {list(df.columns)}")
    print(f"  Containers: {df['new_container_id'].nunique()}")
    log_memory("After loading data")
else:
    print("❌ Failed to load data")

# STEP 3a: Chronological Split + Normalize (Training Stats Only)

In [ ]:
if df is not None:
    logger.info("Splitting data chronologically (60/20/20)...")

    df.sort_values('timestamp', inplace=True)
    df.reset_index(drop=True, inplace=True)

    n = len(df)
    train_end = int(n * 0.6)
    val_end   = int(n * 0.8)

    train_df = df.iloc[:train_end].copy()
    val_df   = df.iloc[train_end:val_end].copy()
    test_df  = df.iloc[val_end:].copy()

    del df
    gc.collect()

    logger.info(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")

    # Metric columns only (exclude ID/label columns)
    metric_cols = [
        c for c in train_df.columns
        if c not in ['timestamp', 'cmdb_id', 'new_container_id']
        and train_df[c].dtype in [np.float32, np.float64]
    ]

    # Calculate stats from TRAINING split only — no data leakage
    logger.info("Calculating normalization stats from training split only...")
    train_stats = {}
    for col in metric_cols:
        mean = float(train_df[col].mean())
        std  = float(train_df[col].std())
        train_stats[col] = {'mean': mean, 'std': std if std > 0 else 1.0}

    # Apply training stats to ALL splits
    def normalize_with_train_stats(data: pd.DataFrame, name: str) -> None:
        """Normalize in-place using TRAINING statistics only."""
        for col, s in train_stats.items():
            if col in data.columns:
                data[col] = ((data[col] - s['mean']) / s['std']).astype(np.float32)
        logger.info(f"  {name}: normalized {len(train_stats)} columns using training stats")

    normalize_with_train_stats(train_df, 'TRAIN')
    normalize_with_train_stats(val_df,   'VAL')
    normalize_with_train_stats(test_df,  'TEST')

    train_norm = train_df
    val_norm   = val_df
    test_norm  = test_df

    print("✓ Split and normalized (training stats only — no leakage)")
    print(f"  Train: {len(train_norm):,} rows | Val: {len(val_norm):,} | Test: {len(test_norm):,}")
    log_memory("After normalization")

# STEP 3b: Feature Engineering (Lag + Rolling per Container)

In [ ]:
if train_norm is not None:
    logger.info("=" * 70)
    logger.info("STEP 3b: FEATURE ENGINEERING")
    logger.info("=" * 70)

    target_columns = [
        'container_cpu_usage_seconds_total',
        'container_memory_usage_bytes',
        'container_memory_working_set_bytes',
        'container_memory_rss'
    ]

    def engineer_features(data: pd.DataFrame, split_name: str) -> pd.DataFrame:
        """
        Add lag diff and rolling features grouped by container.
        Uses groupby so features never bleed across container boundaries.
        """
        logger.info(f"Engineering features for {split_name}...")

        if 'new_container_id' not in data.columns:
            logger.warning(f"  {split_name}: no new_container_id — skipping feature engineering")
            return data

        available = [c for c in target_columns if c in data.columns]
        if not available:
            logger.warning(f"  {split_name}: no target columns found")
            return data

        data = data.sort_values(['new_container_id', 'timestamp']).reset_index(drop=True)

        grp = data.groupby('new_container_id')

        for col in available:
            # Lag difference features (within each container)
            for lag in [1, 2, 3]:
                data[f'{col}_DIFF_{lag}'] = (
                    grp[col].diff(lag).fillna(0).astype(np.float32)
                )
            # Rolling mean (within each container)
            data[f'{col}_ROLLING_MEAN_3'] = (
                grp[col]
                .transform(lambda x: x.rolling(3, min_periods=1).mean())
                .astype(np.float32)
            )
            # Rolling std (within each container)
            data[f'{col}_ROLLING_STD_3'] = (
                grp[col]
                .transform(lambda x: x.rolling(3, min_periods=1).std().fillna(0))
                .astype(np.float32)
            )

        logger.info(f"  {split_name}: {len(data.columns)} total columns")
        return data

    train_feat = engineer_features(train_norm, 'TRAIN')
    val_feat   = engineer_features(val_norm,   'VAL')
    test_feat  = engineer_features(test_norm,  'TEST')

    # Save to disk — create merged_path only if needed
    merged_path.mkdir(parents=True, exist_ok=True)

    train_feat.to_csv(merged_path / 'train_data_with_features.csv', index=False)
    del train_feat, train_norm
    gc.collect()
    logger.info("  ✓ Train saved and cleared from RAM")

    val_feat.to_csv(merged_path / 'val_data_with_features.csv', index=False)
    del val_feat, val_norm
    gc.collect()
    logger.info("  ✓ Val saved and cleared from RAM")

    test_feat.to_csv(merged_path / 'test_data_with_features.csv', index=False)
    del test_feat, test_norm
    gc.collect()
    logger.info("  ✓ Test saved and cleared from RAM")

    print("✓ Feature engineering complete")
    print(f"  Files saved to: {merged_path}")
    log_memory("After feature engineering")

# STEP 4: Sliding Window Sequence Generation

In [ ]:
from numpy.lib.stride_tricks import sliding_window_view

class UltraFastSequenceGeneratorOptimized:
    """
    Fully vectorized sliding-window sequence generator.

    Key design:
      - sliding_window_view  : creates all X windows at once (no Python loop over positions)
      - Per-container files  : each container saves its own temp .npy (small, fast writes)
      - open_memmap merge    : final .npy written in a single O(n) pass
      - y_indices            : pre-computed once, not inside any loop
      Speedup vs loop-based: ~100x for large datasets
    """

    def __init__(self, lookback_window: int = 240, max_horizon: int = 10, batch_size: int = 2000):
        self.lookback_window = lookback_window
        self.max_horizon     = max_horizon
        self.batch_size      = batch_size   # kept for API compat, not used in vectorized path
        self.target_metrics  = [
            'container_cpu_usage_seconds_total',
            'container_memory_usage_bytes',
            'container_memory_working_set_bytes',
            'container_memory_rss'
        ]
        self.feature_cols = None

    def determine_feature_columns(self, df) -> list:
        exclude = {'timestamp', 'case_source', 'cmdb_id', 'new_container_id'}
        cols = [c for c in df.columns
                if c not in exclude
                and df[c].dtype in [np.float64, np.float32, int]]
        logger.info(f"Feature columns: {len(cols)}")
        return cols

    def process_and_save_sequences(self, csv_path: str, dataset_name: str, output_dir: str) -> bool:
        """
        Vectorized sequence generation:
          1. For each container: sliding_window_view creates all X windows in one numpy call
          2. NaN filtering done with boolean masks (no per-row Python checks)
          3. Each container writes its own small temp files
          4. Final merge uses open_memmap for O(n) disk writes
        """
        df            = None
        feature_data  = None
        container_ids = None

        try:
            logger.info(f"\nGenerating sequences for {dataset_name}...")
            df = pd.read_csv(csv_path)
            logger.info(f"  {len(df):,} rows x {len(df.columns)} columns")

            if self.feature_cols is None:
                self.feature_cols = self.determine_feature_columns(df)

            if 'new_container_id' not in df.columns or 'timestamp' not in df.columns:
                logger.error("Missing new_container_id or timestamp")
                return False

            df.sort_values(['new_container_id', 'timestamp'], inplace=True)
            df.reset_index(drop=True, inplace=True)

            output_path = Path(output_dir)
            output_path.mkdir(exist_ok=True)

            # Pre-compute y column indices once
            y_indices = [self.feature_cols.index(m)
                         for m in self.target_metrics if m in self.feature_cols]
            n_feat    = len(self.feature_cols)

            feature_data  = df[self.feature_cols].values.astype(np.float32)
            container_ids = df['new_container_id'].values

            del df
            df = None
            gc.collect()

            unique_containers = np.unique(container_ids)
            logger.info(f"Processing {len(unique_containers)} containers "
                        f"(lookback={self.lookback_window}, horizons=1-{self.max_horizon})...")

            # Track per-horizon counts and which containers contributed
            sequence_counts   = {h: 0 for h in range(1, self.max_horizon + 1)}
            container_indices = {h: [] for h in range(1, self.max_horizon + 1)}

            for idx, container_id in enumerate(unique_containers):
                if (idx + 1) % 5 == 0:
                    total = sum(sequence_counts.values())
                    logger.info(f"  Container {idx+1}/{len(unique_containers)} "
                                f"({total:,} sequences so far)")

                mask   = container_ids == container_id
                c_idx  = np.where(mask)[0]
                n_rows = len(c_idx)

                min_rows = self.lookback_window + self.max_horizon + 1
                if n_rows < min_rows:
                    continue

                cont_data = feature_data[c_idx].astype(np.float32)  # (n_rows, n_feat)

                # VECTORIZED: create all X windows at once using stride tricks
                # sliding_window_view output shape: (n_rows-lw+1, 1, lw, n_feat) -> squeeze -> (n_valid_w, lw, n_feat)
                n_windows = n_rows - self.lookback_window - self.max_horizon
                if n_windows <= 0:
                    del cont_data
                    continue

                X_windows = sliding_window_view(
                    cont_data, (self.lookback_window, n_feat)
                )[:n_windows, 0, :, :]  # view: (n_windows, lookback, n_feat)

                # VECTORIZED NaN filter on X windows
                nan_in_X   = np.isnan(X_windows).any(axis=(1, 2))   # (n_windows,)
                valid_pos  = np.where(~nan_in_X)[0]                  # positions with clean X

                if len(valid_pos) == 0:
                    del cont_data, X_windows
                    gc.collect()
                    continue

                # Copy only the valid windows (necessary to own the data)
                X_clean = np.ascontiguousarray(X_windows[valid_pos], dtype=np.float32)
                del X_windows, nan_in_X
                gc.collect()

                for horizon in range(1, self.max_horizon + 1):
                    # Target row index in cont_data for each valid position
                    tgt_rows   = valid_pos + self.lookback_window + horizon - 1
                    in_bounds  = tgt_rows < n_rows

                    X_h = X_clean[in_bounds]
                    y_h = cont_data[tgt_rows[in_bounds]][:, y_indices].astype(np.float32)

                    # NaN filter on y
                    nan_in_y = np.isnan(y_h).any(axis=1)
                    if nan_in_y.any():
                        X_h = X_h[~nan_in_y]
                        y_h = y_h[~nan_in_y]

                    if len(X_h) == 0:
                        del X_h, y_h
                        continue

                    c_h = np.full(len(X_h), container_id, dtype=object)

                    # Save this container slice as a temp file
                    np.save(str(output_path / f"_tmp_h{horizon}_X_{dataset_name}_{idx}.npy"), X_h)
                    np.save(str(output_path / f"_tmp_h{horizon}_y_{dataset_name}_{idx}.npy"), y_h)
                    np.save(str(output_path / f"_tmp_h{horizon}_c_{dataset_name}_{idx}.npy"), c_h)

                    sequence_counts[horizon]   += len(X_h)
                    container_indices[horizon].append(idx)

                    del X_h, y_h, c_h

                del cont_data, X_clean
                gc.collect()

            total = sum(sequence_counts.values())
            logger.info(f"Total valid sequences: {total:,}")

            # Merge per-container temp files into final .npy using memmap
            logger.info("Merging into final .npy files (memmap)...")
            n_containers_processed = len(unique_containers)

            for horizon in range(1, self.max_horizon + 1):
                if sequence_counts[horizon] == 0:
                    continue
                self._merge_container_files(
                    horizon, dataset_name, output_path,
                    container_indices[horizon], sequence_counts[horizon]
                )
                meta = {
                    'horizon':         horizon,
                    'dataset':         dataset_name,
                    'n_sequences':     sequence_counts[horizon],
                    'lookback_window': self.lookback_window,
                    'n_features':      len(self.feature_cols),
                    'feature_cols':    self.feature_cols,
                    'target_metrics':  self.target_metrics,
                    'X_shape':         [sequence_counts[horizon], self.lookback_window, len(self.feature_cols)],
                    'y_shape':         [sequence_counts[horizon], len(self.target_metrics)],
                }
                with open(output_path / f"sequences_horizon_{horizon}_metadata_{dataset_name}.json", 'w') as f:
                    json.dump(meta, f, indent=2)
                logger.info(f"  Horizon {horizon}: {sequence_counts[horizon]:,} sequences")

            return True

        except Exception as e:
            logger.error(f"Error processing {dataset_name}: {e}")
            import traceback
            logger.error(traceback.format_exc())
            return False

        finally:
            del df, feature_data, container_ids
            gc.collect()

    def _merge_container_files(self, horizon, dataset_name, output_path,
                                container_idx_list, n_sequences):
        """
        Merge per-container temp files into final .npy using open_memmap.
        Each sequence is written exactly once -> O(n) total disk I/O.
        """
        n_feat    = len(self.feature_cols)
        n_targets = len(self.target_metrics)

        X_path = str(output_path / f"sequences_horizon_{horizon}_X_{dataset_name}.npy")
        y_path = str(output_path / f"sequences_horizon_{horizon}_y_{dataset_name}.npy")
        c_path = str(output_path / f"sequences_horizon_{horizon}_containers_{dataset_name}.npy")

        X_mm = np.lib.format.open_memmap(
            X_path, mode='w+', dtype=np.float32,
            shape=(n_sequences, self.lookback_window, n_feat))
        y_mm = np.lib.format.open_memmap(
            y_path, mode='w+', dtype=np.float32,
            shape=(n_sequences, n_targets))
        c_all = []

        offset = 0
        for c_idx in container_idx_list:
            X_tmp = output_path / f"_tmp_h{horizon}_X_{dataset_name}_{c_idx}.npy"
            y_tmp = output_path / f"_tmp_h{horizon}_y_{dataset_name}_{c_idx}.npy"
            c_tmp = output_path / f"_tmp_h{horizon}_c_{dataset_name}_{c_idx}.npy"

            if not X_tmp.exists():
                continue

            X_b = np.load(str(X_tmp))
            y_b = np.load(str(y_tmp))
            c_b = np.load(str(c_tmp), allow_pickle=True)
            n   = len(X_b)

            X_mm[offset:offset + n] = X_b
            y_mm[offset:offset + n] = y_b
            c_all.extend(c_b.tolist())
            offset += n

            del X_b, y_b, c_b
            X_tmp.unlink()
            y_tmp.unlink()
            c_tmp.unlink()
            gc.collect()

        del X_mm, y_mm
        np.save(c_path, np.array(c_all, dtype=object))
        del c_all
        gc.collect()
        logger.info(f"    h{horizon}: merged {offset} sequences")

    def diagnose(self, merged_path, sequences_path) -> None:
        merged_path    = Path(merged_path)
        sequences_path = Path(sequences_path)
        print("\n" + "="*70)
        print("DIAGNOSING SEQUENCE GENERATION")
        print("="*70)
        for name, f in [('train', merged_path / 'train_data_with_features.csv'),
                        ('val',   merged_path / 'val_data_with_features.csv'),
                        ('test',  merged_path / 'test_data_with_features.csv')]:
            print(f"  {name}: exists={f.exists()}")
            if f.exists():
                s = pd.read_csv(f, nrows=2)
                print(f"    cols={list(s.columns)[:5]} | has_id={'new_container_id' in s.columns}")
        print(f"  sequences dir exists: {sequences_path.exists()}")
        if sequences_path.exists():
            files = list(sequences_path.iterdir())
            print(f"  files: {len(files)}")
            for f in sorted(files)[:10]:
                print(f"    {f.name}")

print("✓ UltraFastSequenceGeneratorOptimized (vectorized) ready")

In [ ]:
logger.info("\n" + "#"*70)
logger.info("STEP 4: SEQUENCE GENERATION")
logger.info("#"*70)

generator = UltraFastSequenceGeneratorOptimized(lookback_window=240, max_horizon=10, batch_size=500)

datasets = [
    ('train', merged_path / 'train_data_with_features.csv'),
    ('val',   merged_path / 'val_data_with_features.csv'),
    ('test',  merged_path / 'test_data_with_features.csv'),
]

all_success = True
for data_name, csv_file in datasets:
    logger.info(f"\nProcessing {data_name.upper()}...")
    success = generator.process_and_save_sequences(str(csv_file), data_name, str(sequences_path))
    if not success:
        logger.error(f"❌ Failed to generate sequences for {data_name}")
        all_success = False
    log_memory(f"After {data_name} sequences")

if all_success:
    print("\n✓ All sequences generated successfully")
else:
    print("\n⚠ Some datasets failed — check logs above")

# STEP 7: Verify Sequences

In [ ]:
print("="*70)
print("VERIFYING SEQUENCES")
print("="*70)

try:
    X_train = np.load(sequences_path / 'sequences_horizon_1_X_train.npy')
    y_train = np.load(sequences_path / 'sequences_horizon_1_y_train.npy')

    print(f"\nTraining Data (Horizon 1):")
    print(f"  X shape: {X_train.shape}")
    print(f"  y shape: {y_train.shape}")

    npy_files = list(sequences_path.glob('*.npy'))
    json_files = list(sequences_path.glob('*.json'))

    print(f"\n  Total .npy files: {len(npy_files)}")
    print(f"  Total .json files: {len(json_files)}")

    print("\n" + "="*70)
    print("✅ SEQUENCES GENERATED SUCCESSFULLY!")
    print("="*70)

    final_mem = log_memory("Final")
    print(f"\n💾 Memory Summary:")
    print(f"  Initial: {initial_mem:.1f} MB")
    print(f"  Final:   {final_mem:.1f} MB")
    print(f"  Peak:    ~{final_mem:.1f} MB")

except FileNotFoundError as e:
    print(f"❌ Error: {e}")
    print("Sequences were not generated. Running diagnostics...\n")
    generator.diagnose(merged_path, sequences_path)

# STEP 5: Convert .npy Sequences to CSV

In [ ]:
import numpy as np
import pandas as pd
import gc
from pathlib import Path

# Local sequences directory — all CSVs saved here, no new folders
sequences_path = Path(r'C:/Users/rasen/Documents/Claude/Projects/Module2/module2/data/sequences')

target_metrics = [
    'container_cpu_usage_seconds_total',
    'container_memory_usage_bytes',
    'container_memory_working_set_bytes',
    'container_memory_rss'
]

print("=" * 70)
print("STEP 5: CONVERTING .NPY SEQUENCES TO CSV")
print(f"  Directory: {sequences_path}")
print("=" * 70)

# Find all .npy files dynamically
npy_files = sorted(sequences_path.glob('*.npy'))
print(f"Found {len(npy_files)} .npy files
")

converted = 0
skipped   = 0

for npy_file in npy_files:
    csv_file = npy_file.with_suffix('.csv')  # same name, .csv extension

    arr = np.load(str(npy_file), allow_pickle=True)
    name = npy_file.stem  # e.g. sequences_horizon_1_X_train

    # ── containers (1D: n_seq) ─────────────────────────────────────────────
    if '_containers_' in name:
        df = pd.DataFrame({'container_id': arr})
        df.to_csv(str(csv_file), index=False)
        print(f"  containers  {name}: {arr.shape[0]} rows -> {csv_file.name}")
        del df

    # ── y (2D: n_seq x 4 targets) ─────────────────────────────────────────
    elif name.split('_')[2] == 'y' or ('_y_' in name):
        pd.DataFrame(arr, columns=target_metrics).to_csv(str(csv_file), index=False)
        print(f"  y           {name}: {arr.shape} -> {csv_file.name}")

    # ── X (3D: n_seq x lookback x n_features) -> flattened ────────────────
    elif '_X_' in name:
        n_seq, lookback, n_feat = arr.shape
        col_names = [f"t{t}_f{f}" for t in range(lookback) for f in range(n_feat)]
        X_flat = arr.reshape(n_seq, lookback * n_feat)
        del arr
        gc.collect()
        pd.DataFrame(X_flat, columns=col_names).to_csv(str(csv_file), index=False)
        del X_flat
        gc.collect()
        print(f"  X           {name}: ({n_seq} x {lookback} x {n_feat}) -> {csv_file.name}")
        converted += 1
        continue

    del arr
    gc.collect()
    converted += 1

print(f"
Done: {converted} CSV files saved in {sequences_path}")